In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn
import torchvision.transforms.functional as TF
from skimage.metrics import structural_similarity as ssim

2026-03-12 14:05:03.901711: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
def unet_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             ):

    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label[..., 0]

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)

        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    def input_pipeline_downstream(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):

        feature_description = {
            'hres': tf.io.FixedLenFeature([7*hres_size_4x*hres_size_4x], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size_4x, hres_size_4x])
            hres = tf.cast(hres, tf.float32)
            hres = hres*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])

            return hres, label[..., 0]

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)

        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    def calculate_batch_psnr(pred_batch, target_batch, max_val=1.0):
        # pred, target: [B, C, H, W]
        mse = F.mse_loss(pred_batch, target_batch, reduction='none')  # [B, C, H, W]
        mse = mse.view(mse.shape[0], mse.shape[1], -1).mean(dim=2)  # [B, C]
        psnr = 10 * torch.log10(max_val**2 / mse)  # [B, C]
        return psnr.mean(dim=1)  # mean over channels → [B]

    def calculate_batch_ssim(pred_batch, target_batch):
        # pred, target: [B, C, H, W] → loop over batch
        B, C, H, W = pred_batch.shape
        ssim_scores = []

        for i in range(B):
            pred = pred_batch[i].cpu().numpy()
            target = target_batch[i].cpu().numpy()
            # Compute per-band SSIM and average
            ssim_per_band = [ssim(pred[c], target[c], data_range=1.0) for c in range(C)]
            ssim_scores.append(np.mean(ssim_per_band))

        return torch.tensor(ssim_scores)  # shape: [B]

    gt_loader = input_pipeline_downstream_sr(finetune_tfrecords, 20, num_training, num_test, is_shuffle=False, is_repeat=False)
    sr_loader = input_pipeline_downstream(sr_tfrecords, 20, num_training, num_test, is_shuffle=False, is_repeat=False)
    zipped_dataset = tf.data.Dataset.zip((gt_loader, sr_loader))

    total_psnr = []
    total_ssim = []
    for (_, hr_gt, _), (hr, label) in zipped_dataset:
        batch_size = hr_gt.shape[0]

        hr_gt = torch.from_numpy(hr_gt.numpy().astype('float32'))
        hr = torch.from_numpy(hr.numpy().astype('float32'))
        # print(hr.shape)
        # print(hr_gt.shape)

        hr = F.interpolate(hr, size=(hres_size, hres_size), mode='bilinear', align_corners=False)
        # hr_gt = F.interpolate(hr_gt, size=(hres_size_4x, hres_size_4x), mode='bilinear')
        # print(hr.shape)

        hr_gt = torch.clamp(hr_gt, 0, 1)
        hr = torch.clamp(hr, 0, 1)

        # Compute PSNR & SSIM for this batch
        psnr_batch = calculate_batch_psnr(hr, hr_gt)  # [B]
        ssim_batch = calculate_batch_ssim(hr, hr_gt)  # [B]

        total_psnr.append(psnr_batch)
        total_ssim.append(ssim_batch)

        # break

        # for i in range(batch_size):
        #     fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        #     hr_gt_show = np.transpose(hr_gt[i].numpy(), (1, 2, 0))
        #     axes[0].imshow(hr_gt_show[:, :, [3,2,1]]*2)
        #     axes[0].set_title('GT')
        #     axes[0].axis('off')

        #     output_show = np.transpose(hr[i], (1, 2, 0))
        #     axes[1].imshow(output_show[:, :, [3,2,1]]*2)
        #     # axes[1].imshow(output_show[:, :, [3, 2, 1]])
        #     axes[1].set_title('High-Resolution')
        #     axes[1].axis('off')

        #     axes[2].imshow(label[i])
        #     axes[2].set_title('Output')
        #     axes[2].axis('off')

        # break

    # Concatenate and compute overall average
    total_psnr = torch.cat(total_psnr).mean().item()
    total_ssim = torch.cat(total_ssim).mean().item()

    print(f"Average PSNR: {total_psnr} dB")
    print(f"Average SSIM: {total_ssim}")






In [3]:
dataset_name = 'M2L8_CHM'
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_UPSR_x16.tfrecords']

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1408
num_training = 1126
class_num=2
start_class=0
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']

for sr_tfrecords in output_tfrecords_files[-1:]:
    print(sr_tfrecords)
    unet_run(lres_size ,
            hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             )

In [5]:
dataset_name = 'M2L8_Urban'
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_UPSR_x16.tfrecords']

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
class_num=11
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']

for sr_tfrecords in output_tfrecords_files[-1:]:
    print(sr_tfrecords)
    unet_run(lres_size ,
            hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             )

In [7]:
dataset_name = 'M2L8_CDL'
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CDL_UPSR_x16.tfrecords']

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
class_num=5
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']

for sr_tfrecords in output_tfrecords_files:
    print(sr_tfrecords)
    unet_run(lres_size ,
            hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             )

In [9]:
dataset_name = 'M2L8_GPP'
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_GPP_UPSR_x16.tfrecords']

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 755
num_training = 604
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']

for sr_tfrecords in output_tfrecords_files:
    print(sr_tfrecords)
    unet_run(lres_size ,
             hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             )

In [11]:
dataset_name = 'M2L8_River'
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_River_UPSR_x16.tfrecords']

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1065
num_training = 852
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']

for sr_tfrecords in output_tfrecords_files:
    print(sr_tfrecords)
    unet_run(lres_size ,
             hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    sr_tfrecords
             )